# Plotting Notebook
This notebook is to help you plot your PHREEQC output data.

## Run this cell first to import necessary modules and setup plotting functions

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go


def load_phreeqc_output(filepath, sep="\t", **kwargs):
    """Load a tab-delimited PHREEQC .sel/.dat file into a DataFrame.

    Strips whitespace from header names and drops any empty/unnamed
    columns produced by trailing delimiters in the source file.

    The rows are dropped before the pH increases, so the DataFrame will
    only contain the rows after the pH increases.
    """
    df = pd.read_csv(filepath, sep=sep, **kwargs)
    df.columns = df.columns.str.strip()
    df = df.loc[:, ~df.columns.str.match(r"^Unnamed(:.*)?$|^$")]

    return df


def concentration_plot(data, x_col, y_col, drop_rows=None):
    """Plot concentration data from a PHREEQC output DataFrame.

    Parameters:
    - data: DataFrame containing the PHREEQC output.
    - x_col: Column name for the x-axis (e.g., pH).
    - y_col: list of column names for the y-axis (e.g., species concentrations).

    Returns:
    - A Plotly figure object.
    """
    if drop_rows is not None:
        data = data.drop(drop_rows).reset_index(drop=True)

    fig = go.Figure()
    for col in y_col:
        fig.add_trace(
            go.Scatter(x=data[x_col], y=data[col], mode="lines+markers", name=col)
        )
    fig.update_layout(xaxis_title=x_col, yaxis_title="Concentration (mol/L)")

    return fig


def percent_plot(data, x_col, y_col, total_conc, drop_rows=None):
    """Plot percent data from a PHREEQC output DataFrame.

    Parameters:
    - data: DataFrame containing the PHREEQC output.
    - x_col: Column name for the x-axis (e.g., pH).
    - y_col: list of column names for the y-axis (e.g., species concentrations).
    - total_conc: Column name for the total concentration to calculate percentages.

    Returns:
    - A Plotly figure object.
    """
    if drop_rows is not None:
        data = data.drop(drop_rows).reset_index(drop=True)

    fig = go.Figure()
    for col in y_col:
        percent = (data[col] / total_conc) * 100
        fig.add_trace(
            go.Scatter(x=data[x_col], y=percent, mode="lines+markers", name=col)
        )
    fig.update_layout(xaxis_title=x_col, yaxis_title="Percent (%)")

    return fig

# Load your data
Replace `example.sel` with your filename

In [2]:
data = load_phreeqc_output("example2.sel")
print("You can plot the following columns:")
print(data.columns.tolist())
data

You can plot the following columns:
['pH', 'm_Hfo_sOCd+', 'm_Hfo_wOCd+']


,pH,m_Hfo_sOCd+,m_Hfo_wOCd+
0,7.00000,0.000000e+00,0.000000e+00
1,5.59431,1.278100e-06,2.305900e-08
2,3.00000,8.059400e-12,1.416200e-13
3,3.50000,8.967600e-11,1.576200e-12
4,4.00000,9.425100e-10,1.656800e-11
5,4.50000,9.631800e-09,1.693500e-10
6,5.00000,9.621800e-08,1.694700e-09
7,5.50000,8.740700e-07,1.564000e-08
8,6.00000,4.639200e-06,8.992000e-08
9,6.50000,8.717400e-06,1.857100e-07


# Concentration vs pH

In [3]:
# data to plot
xaxis = "pH"
yaxis = ["m_Hfo_sOCd+", "m_Hfo_wOCd+"]

# Rows to drop
drop_rows = [0, 1]

fig = concentration_plot(data, xaxis, yaxis, drop_rows=drop_rows)
fig.show()

# Percent vs pH
You need to specify a total concentration in addition to the data to plot

In [4]:
# Total concentration for percent plot
total_conc = 1e-5

# data to plot
xaxis = "pH"
yaxis = ["m_Hfo_sOCd+", "m_Hfo_wOCd+"]

# Rows to drop
drop_rows = [0, 1]

fig = percent_plot(data, xaxis, yaxis, total_conc, drop_rows=drop_rows)
fig.show()